# Import Libraries 

In [21]:
import ee


print("Authenticating with Google Earth Engine...")
ee.Authenticate()

print("Initializing project...")
ee.Initialize(project="agriculture-drought-assesment")

Authenticating with Google Earth Engine...
Initializing project...


In [22]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [23]:
import importlib

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn import linear_model
from data import indices, load_CHIRPS, load_ERA5, load_S2, utils, ee_utils
from visualisation import index_plotter
from features import engineering

importlib.reload(indices)
importlib.reload(load_S2)
importlib.reload(load_ERA5)
importlib.reload(load_CHIRPS)
importlib.reload(utils)
importlib.reload(index_plotter)
importlib.reload(engineering)

<module 'features.engineering' from '/home/luke/projects/agriculture-analysis/src/features/engineering.py'>

# Prepare DataFrames

In [24]:
region = utils.get_region()
i_date = "2018-01-01"
f_date = "2025-12-31"

In [25]:

    print(f"Loading Sentinel-2 dataset for periods {i_date} to {f_date}...")
    s2_clouded = load_S2.get_s2_data(region, i_date, f_date)

    print(f"Loading ERA5 Land data for periods {i_date} to {f_date}...")
    era5_data = load_ERA5.get_era5_data(region, i_date, f_date)

    print(f"Loading CHIRPS Climate Data for periods {i_date} to {f_date}...")
    chirps_data = load_CHIRPS.get_chirps_data(region, i_date, f_date)

    s2_clear_sky = s2_clouded.map(load_S2.s2_clear_sky)

    reducer_fn = utils.create_reducer(region, scale=10)
    ndvi_df = indices.get_index_df(s2_clear_sky, indices.NDVI, reducer_fn)
    ndre_df = indices.get_index_df(s2_clear_sky, indices.NDRE, reducer_fn)
    ndwi_df = indices.get_index_df(s2_clear_sky, indices.NDWI, reducer_fn)

    reducer_fn = utils.create_reducer(region, scale=11132)
    slvw_df = load_ERA5.get_soil_levels_df(era5_data, reducer_fn, "water")
    slt_df = load_ERA5.get_soil_levels_df(era5_data, reducer_fn, "temp")
    temp_2m_df = load_ERA5.get_temp_2m_df(era5_data, reducer_fn)
    ssrd_df = load_ERA5.get_ssrd(era5_data, reducer_fn)

    reducer_fn = utils.create_reducer(region, scale=5566)
    perc_df = load_CHIRPS.get_precipitation_df(chirps_data, reducer_fn)

    grouped = indices.apply_groupby_date(ndwi_df, ndvi_df, ndre_df)
    ndwi_df_grouped = grouped[0]
    ndvi_df_grouped = grouped[1]
    ndre_df_grouped = grouped[2]

    master_df = utils.merge_with_master(
        ndvi_df_grouped,
        ndwi_df_grouped,
        ndre_df_grouped,
        slvw_df,
        slt_df,
        temp_2m_df,
        ssrd_df,
        perc_df,
    )
    print("Done!")



Loading Sentinel-2 dataset for periods 2018-01-01 to 2025-12-31...
Loading ERA5 Land data for periods 2018-01-01 to 2025-12-31...
Loading CHIRPS Climate Data for periods 2018-01-01 to 2025-12-31...
Creating NDVI DataFrame...
Creating NDRE DataFrame...
Creating NDWI DataFrame...
Creating volumetric_soil_water_layer_1 DataFrame...
Creating volumetric_soil_water_layer_2 DataFrame...
Creating volumetric_soil_water_layer_3 DataFrame...
Creating volumetric_soil_water_layer_4 DataFrame...
Concatinated water levels DataFrame.
Creating soil_temperature_level_1 DataFrame...
Creating soil_temperature_level_2 DataFrame...
Creating soil_temperature_level_3 DataFrame...
Creating soil_temperature_level_4 DataFrame...
Concatinated temp levels DataFrame.
Creating temperature_2m DataFrame...
Creating temperature_2m_min DataFrame...
Creating temperature_2m_max DataFrame...
Creating surface_solar_radiation_downwards_sum DataFrame...
Creating precipitation DataFrame...
Done!


In [26]:
non_spectral_df = utils.merge_with_master(slvw_df, slt_df, temp_2m_df, ssrd_df, perc_df)
non_spectral_df.to_csv("../data/raw/non_spectral_df.csv")

In [27]:
spectral_df = utils.merge_with_master(ndvi_df, ndwi_df, ndre_df)
spectral_df.to_csv("../data/raw/spectral_df.csv")

In [28]:
grouped = indices.apply_groupby_date(ndwi_df, ndvi_df, ndre_df)
ndwi_df_grouped = grouped[0]
ndvi_df_grouped = grouped[1]
ndre_df_grouped = grouped[2]


In [29]:
grouped_spectral_df = utils.merge_with_master(ndvi_df_grouped, ndwi_df_grouped, ndre_df_grouped)
grouped_spectral_df.to_csv("../data/raw/grouped_spectral_df.csv")